In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import json
import shap
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

In [2]:
PROJECT_ROOT = Path("..")

FEATURE_DATA_PATH = PROJECT_ROOT / "01_data" / "features" / "sales_features.csv"

FINAL_MODEL_PATH = PROJECT_ROOT / "04_models" / "final_sales_forecasting_model.pkl"
FINAL_FEATURE_LIST_PATH = PROJECT_ROOT / "04_models" / "final_feature_columns.json"
FINAL_MODEL_INFO_PATH = PROJECT_ROOT / "04_models" / "final_model_info.json"

MODEL_RESULTS_DIR = PROJECT_ROOT / "05_outputs" / "model_results"
FIGURES_DIR = PROJECT_ROOT / "05_outputs" / "figures"

MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(FEATURE_DATA_PATH)
print(FINAL_MODEL_PATH)

..\01_data\features\sales_features.csv
..\04_models\final_sales_forecasting_model.pkl


In [3]:
final_model = joblib.load(FINAL_MODEL_PATH)

with open(FINAL_FEATURE_LIST_PATH, "r") as f:
    feature_cols = json.load(f)

with open(FINAL_MODEL_INFO_PATH, "r") as f:
    final_model_info = json.load(f)

print("Final model loaded.")
print("Model info:")
display(final_model_info)

print("\nNumber of features:", len(feature_cols))
print(feature_cols)

Final model loaded.
Model info:


{'model_name': 'Final_LightGBM_V1_Tuned_1',
 'base_model': 'LightGBM',
 'target_type': 'log',
 'trained_on_train_validation': True,
 'feature_set': 'sales_features_v1',
 'train_end_date': '2021-08-19T00:00:00.000000',
 'validation_end_date': '2021-10-26T00:00:00.000000',
 'number_of_features': 29,
 'test_wape': 0.4529717777145094,
 'test_wape_percent': 45.29717777145094,
 'best_params': {'n_estimators': 300,
  'learning_rate': 0.05,
  'num_leaves': 31,
  'max_depth': -1,
  'min_child_samples': 20,
  'subsample': 0.8,
  'colsample_bytree': 0.8,
  'reg_alpha': 0,
  'reg_lambda': 0}}


Number of features: 29
['sku_encoded', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'moq_order', 'qty_lag_1', 'qty_lag_2', 'qty_lag_3', 'qty_lag_7', 'qty_lag_14', 'qty_rolling_mean_3', 'qty_rolling_std_3', 'qty_rolling_min_3', 'qty_rolling_max_3', 'qty_rolling_mean_7', 'qty_rolling_std_7', 'qty_rolling_min_7', 'qty_rolling_max_7', 'qty_rolling_mean_14', 'qty_rolling_std_14', 'qty_rolling_min_14', 'qty_rolling_max_14', 'sku_expanding_mean_qty', 'sku_expanding_median_qty', 'sku_expanding_std_qty']


In [4]:
df = pd.read_csv(FEATURE_DATA_PATH)
df["shipped_date"] = pd.to_datetime(df["shipped_date"])

print("Shape:", df.shape)
print("Date range:", df["shipped_date"].min(), "to", df["shipped_date"].max())
print("Unique SKUs:", df["sku"].nunique())

df.head()

Shape: (26484, 36)
Date range: 2021-01-29 00:00:00 to 2022-01-02 00:00:00
Unique SKUs: 180


,shipped_date,sku,sku_encoded,qty,qty_log,year,month,day,day_of_week,week_of_year,quarter,is_weekend,revenue,cost_of_good_sold,moq_order,channel_count,qty_lag_1,qty_lag_2,qty_lag_3,qty_lag_7,qty_lag_14,qty_rolling_mean_3,qty_rolling_std_3,qty_rolling_min_3,qty_rolling_max_3,qty_rolling_mean_7,qty_rolling_std_7,qty_rolling_min_7,qty_rolling_max_7,qty_rolling_mean_14,qty_rolling_std_14,qty_rolling_min_14,qty_rolling_max_14,sku_expanding_mean_qty,sku_expanding_median_qty,sku_expanding_std_qty
0,2021-02-04,089A0E,0,20,3.044522,2021,2,4,3,5,1,0,529.2,428.65,56460,1,300.0,110.0,90.0,60.0,220.0,166.666667,115.902258,90.0,300.0,121.428571,92.992575,40.0,300.0,128.571429,76.445772,40.0,300.0,128.571429,125.0,76.445772
1,2021-02-06,089A0E,0,60,4.110874,2021,2,6,5,5,1,1,1587.6,1285.96,56460,1,20.0,300.0,110.0,190.0,140.0,143.333333,142.945211,20.0,300.0,115.714286,98.464400,20.0,300.0,114.285714,76.732732,20.0,300.0,121.333333,110.0,78.818659
2,2021-02-10,089A0E,0,40,3.713572,2021,2,10,2,6,1,0,1058.4,857.30,56460,1,60.0,20.0,300.0,60.0,150.0,126.666667,151.437556,20.0,300.0,97.142857,94.289322,20.0,300.0,108.571429,77.643876,20.0,300.0,117.500000,100.0,77.674535
3,2021-02-12,089A0E,0,100,4.615121,2021,2,12,4,6,1,0,2646.0,2143.26,56460,1,40.0,60.0,20.0,40.0,190.0,40.000000,20.000000,20.0,60.0,94.285714,95.891804,20.0,300.0,100.714286,78.687726,20.0,300.0,112.941176,90.0,77.521344
4,2021-02-14,089A0E,0,50,3.931826,2021,2,14,6,6,1,1,1323.0,1071.63,56460,1,100.0,40.0,60.0,90.0,60.0,66.666667,30.550505,40.0,100.0,102.857143,92.864469,20.0,300.0,94.285714,74.391303,20.0,300.0,112.222222,95.0,75.268582


In [5]:
unique_dates = np.sort(df["shipped_date"].unique())

train_end_idx = int(len(unique_dates) * 0.6)
val_end_idx = int(len(unique_dates) * 0.8)

train_end_date = unique_dates[train_end_idx]
val_end_date = unique_dates[val_end_idx]

train_df = df[df["shipped_date"] < train_end_date].copy()

val_df = df[
    (df["shipped_date"] >= train_end_date) &
    (df["shipped_date"] < val_end_date)
].copy()

test_df = df[df["shipped_date"] >= val_end_date].copy()

print("Train:", train_df.shape, train_df["shipped_date"].min(), "to", train_df["shipped_date"].max())
print("Validation:", val_df.shape, val_df["shipped_date"].min(), "to", val_df["shipped_date"].max())
print("Test:", test_df.shape, test_df["shipped_date"].min(), "to", test_df["shipped_date"].max())

Train: (15677, 36) 2021-01-29 00:00:00 to 2021-08-17 00:00:00
Validation: (5425, 36) 2021-08-19 00:00:00 to 2021-10-24 00:00:00
Test: (5382, 36) 2021-10-26 00:00:00 to 2022-01-02 00:00:00


In [6]:
X_test = test_df[feature_cols].copy()
y_test_qty = test_df["qty"].copy()

test_pred_log = final_model.predict(X_test)
test_pred_qty = np.expm1(test_pred_log)
test_pred_qty = np.clip(test_pred_qty, 0, None)

prediction_check = test_df[["shipped_date", "sku", "qty"]].copy()
prediction_check["actual_qty"] = prediction_check["qty"]
prediction_check["predicted_qty"] = test_pred_qty
prediction_check["absolute_error"] = abs(
    prediction_check["actual_qty"] - prediction_check["predicted_qty"]
)

prediction_check = prediction_check.drop(columns=["qty"])

prediction_check.head()

,shipped_date,sku,actual_qty,predicted_qty,absolute_error
92,2021-10-26,089A0E,340,85.120237,254.879763
93,2021-10-30,089A0E,690,82.065806,607.934194
94,2021-11-01,089A0E,430,173.302765,256.697235
95,2021-11-05,089A0E,1030,294.237123,735.762877
96,2021-11-07,089A0E,1180,207.889818,972.110182


In [7]:
explainer = shap.TreeExplainer(final_model)

print("SHAP explainer created.")

SHAP explainer created.


In [8]:
SAMPLE_SIZE = min(1000, len(X_test))

X_shap = X_test.sample(n=SAMPLE_SIZE, random_state=42)

print("X_shap shape:", X_shap.shape)
X_shap.head()

X_shap shape: (1000, 29)


,sku_encoded,year,month,day,day_of_week,week_of_year,quarter,is_weekend,moq_order,qty_lag_1,qty_lag_2,qty_lag_3,qty_lag_7,qty_lag_14,qty_rolling_mean_3,qty_rolling_std_3,qty_rolling_min_3,qty_rolling_max_3,qty_rolling_mean_7,qty_rolling_std_7,qty_rolling_min_7,qty_rolling_max_7,qty_rolling_mean_14,qty_rolling_std_14,qty_rolling_min_14,qty_rolling_max_14,sku_expanding_mean_qty,sku_expanding_median_qty,sku_expanding_std_qty
16162,109,2021,11,19,4,46,4,0,284922,495.0,495.0,495.0,693.0,594.0,495.000000,0.000000,495.0,495.0,636.428571,322.610114,198.0,1188.0,509.142857,310.301059,99.0,1188.0,801.223602,693.0,521.843786
4566,30,2021,12,25,5,51,4,1,90988,575.0,598.0,138.0,713.0,23.0,437.000000,259.196836,138.0,598.0,535.571429,224.457017,138.0,828.0,432.071429,262.268056,23.0,828.0,295.953642,276.0,197.008133
22626,153,2021,10,28,3,43,4,0,910416,962.0,1274.0,1794.0,1222.0,1924.0,1343.333333,420.310996,962.0,1794.0,1326.000000,285.211033,962.0,1794.0,1522.857143,312.452053,962.0,2002.0,1608.360000,1586.0,409.891724
16990,115,2021,12,27,0,52,4,0,360456,828.0,736.0,276.0,1104.0,1472.0,613.333333,295.738623,276.0,828.0,959.428571,724.130480,276.0,2484.0,1261.714286,699.189358,276.0,2484.0,843.114286,736.0,663.936214
1672,11,2021,11,1,0,44,4,0,285516,1344.0,252.0,336.0,420.0,504.0,644.000000,607.670964,252.0,1344.0,492.000000,448.249930,84.0,1344.0,438.000000,328.119115,84.0,1344.0,650.823529,504.0,520.025385


In [9]:
shap_values = explainer.shap_values(X_shap)

print("SHAP values calculated.")
print("SHAP values shape:", np.array(shap_values).shape)

SHAP values calculated.
SHAP values shape: (1000, 29)


In [10]:
shap_importance = pd.DataFrame({
    "feature": X_shap.columns,
    "mean_abs_shap_value": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap_value", ascending=False)

SHAP_IMPORTANCE_PATH = MODEL_RESULTS_DIR / "shap_feature_importance.csv"

shap_importance.to_csv(SHAP_IMPORTANCE_PATH, index=False)

print(f"Saved SHAP feature importance to: {SHAP_IMPORTANCE_PATH}")

shap_importance.head(20)

Saved SHAP feature importance to: ..\05_outputs\model_results\shap_feature_importance.csv


,feature,mean_abs_shap_value
18,qty_rolling_mean_7,0.521671
22,qty_rolling_mean_14,0.143968
4,day_of_week,0.121014
9,qty_lag_1,0.097170
14,qty_rolling_mean_3,0.094678
8,moq_order,0.070803
27,sku_expanding_median_qty,0.064120
26,sku_expanding_mean_qty,0.055592
10,qty_lag_2,0.041388
20,qty_rolling_min_7,0.036730


In [11]:
plt.figure(figsize=(10, 6))

shap.summary_plot(
    shap_values,
    X_shap,
    plot_type="bar",
    show=False,
    max_display=20
)

SHAP_BAR_PATH = FIGURES_DIR / "shap_bar_plot.png"
plt.tight_layout()
plt.savefig(SHAP_BAR_PATH, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved SHAP bar plot to: {SHAP_BAR_PATH}")

Saved SHAP bar plot to: ..\05_outputs\figures\shap_bar_plot.png


In [12]:
plt.figure(figsize=(10, 6))

shap.summary_plot(
    shap_values,
    X_shap,
    show=False,
    max_display=20
)

SHAP_SUMMARY_PATH = FIGURES_DIR / "shap_summary_plot.png"
plt.tight_layout()
plt.savefig(SHAP_SUMMARY_PATH, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved SHAP summary plot to: {SHAP_SUMMARY_PATH}")

Saved SHAP summary plot to: ..\05_outputs\figures\shap_summary_plot.png


In [13]:
selected_index = prediction_check["absolute_error"].idxmax()

selected_row_info = prediction_check.loc[selected_index]
selected_x = X_test.loc[[selected_index]]

selected_shap_values = explainer.shap_values(selected_x)

print("Selected prediction:")
display(selected_row_info)

print("\nSelected feature values:")
display(selected_x)

Selected prediction:


shipped_date      2021-11-21 00:00:00
sku                            XDYFFX
actual_qty                      17091
predicted_qty              835.887877
absolute_error           16255.112123
Name: 24378, dtype: object


Selected feature values:


,sku_encoded,year,month,day,day_of_week,week_of_year,quarter,is_weekend,moq_order,qty_lag_1,qty_lag_2,qty_lag_3,qty_lag_7,qty_lag_14,qty_rolling_mean_3,qty_rolling_std_3,qty_rolling_min_3,qty_rolling_max_3,qty_rolling_mean_7,qty_rolling_std_7,qty_rolling_min_7,qty_rolling_max_7,qty_rolling_mean_14,qty_rolling_std_14,qty_rolling_min_14,qty_rolling_max_14,sku_expanding_mean_qty,sku_expanding_median_qty,sku_expanding_std_qty
24378,165,2021,11,21,6,46,4,1,205983,972.0,810.0,729.0,1134.0,405.0,837.0,123.729544,729.0,972.0,891.0,132.272446,729.0,1134.0,781.071429,293.956686,243.0,1377.0,932.00625,729.0,1181.859235


In [14]:
local_explanation = pd.DataFrame({
    "feature": feature_cols,
    "feature_value": selected_x.iloc[0].values,
    "shap_value": selected_shap_values[0]
})

local_explanation["abs_shap_value"] = local_explanation["shap_value"].abs()

local_explanation = local_explanation.sort_values(
    "abs_shap_value", 
    ascending=False
).reset_index(drop=True)

LOCAL_SHAP_PATH = MODEL_RESULTS_DIR / "local_shap_explanation.csv"

local_explanation.to_csv(LOCAL_SHAP_PATH, index=False)

print(f"Saved local SHAP explanation to: {LOCAL_SHAP_PATH}")

local_explanation.head(20)

Saved local SHAP explanation to: ..\05_outputs\model_results\local_shap_explanation.csv


,feature,feature_value,shap_value,abs_shap_value
0,qty_rolling_mean_7,891.000000,0.851198,0.851198
1,qty_rolling_mean_14,781.071429,0.268386,0.268386
2,qty_rolling_mean_3,837.000000,0.132222,0.132222
3,qty_lag_1,972.000000,0.126238,0.126238
4,sku_expanding_median_qty,729.000000,0.080250,0.080250
5,day_of_week,6.000000,0.068817,0.068817
6,sku_expanding_mean_qty,932.006250,0.042711,0.042711
7,qty_rolling_min_7,729.000000,0.041624,0.041624
8,qty_lag_2,810.000000,0.038070,0.038070
9,moq_order,205983.000000,0.035598,0.035598


In [15]:
positive_drivers = local_explanation[local_explanation["shap_value"] > 0].head(10)
negative_drivers = local_explanation[local_explanation["shap_value"] < 0].head(10)

print("Top features increasing the prediction:")
display(positive_drivers)

print("\nTop features decreasing the prediction:")
display(negative_drivers)

Top features increasing the prediction:


,feature,feature_value,shap_value,abs_shap_value
0,qty_rolling_mean_7,891.000000,0.851198,0.851198
1,qty_rolling_mean_14,781.071429,0.268386,0.268386
2,qty_rolling_mean_3,837.000000,0.132222,0.132222
3,qty_lag_1,972.000000,0.126238,0.126238
4,sku_expanding_median_qty,729.000000,0.080250,0.080250
5,day_of_week,6.000000,0.068817,0.068817
6,sku_expanding_mean_qty,932.006250,0.042711,0.042711
7,qty_rolling_min_7,729.000000,0.041624,0.041624
8,qty_lag_2,810.000000,0.038070,0.038070
9,moq_order,205983.000000,0.035598,0.035598



Top features decreasing the prediction:


,feature,feature_value,shap_value,abs_shap_value
11,week_of_year,46.000000,-0.020448,0.020448
12,qty_rolling_max_7,1134.000000,-0.014650,0.014650
14,qty_rolling_std_14,293.956686,-0.011762,0.011762
16,is_weekend,1.000000,-0.008936,0.008936
18,qty_rolling_std_7,132.272446,-0.005715,0.005715
19,qty_lag_14,405.000000,-0.005611,0.005611
20,qty_rolling_max_14,1377.000000,-0.003007,0.003007
21,month,11.000000,-0.002617,0.002617
25,qty_lag_7,1134.000000,-0.001182,0.001182


In [16]:
top_global_features = shap_importance.head(10).copy()

business_summary = {
    "final_model": final_model_info.get("model_name"),
    "target_type": final_model_info.get("target_type"),
    "test_wape_percent": final_model_info.get("test_wape_percent"),
    "top_1_feature": top_global_features.iloc[0]["feature"],
    "top_2_feature": top_global_features.iloc[1]["feature"],
    "top_3_feature": top_global_features.iloc[2]["feature"],
    "shap_sample_size": SAMPLE_SIZE,
    "business_interpretation": (
        "The model mainly relies on historical SKU demand patterns, such as recent lag values, "
        "rolling averages, and SKU-level historical statistics, to forecast future quantity."
    )
}

business_summary_df = pd.DataFrame([business_summary])

BUSINESS_SHAP_SUMMARY_PATH = MODEL_RESULTS_DIR / "business_shap_summary.csv"

business_summary_df.to_csv(BUSINESS_SHAP_SUMMARY_PATH, index=False)

print(f"Saved business SHAP summary to: {BUSINESS_SHAP_SUMMARY_PATH}")

business_summary_df

Saved business SHAP summary to: ..\05_outputs\model_results\business_shap_summary.csv


,final_model,target_type,test_wape_percent,top_1_feature,top_2_feature,top_3_feature,shap_sample_size,business_interpretation
0,Final_LightGBM_V1_Tuned_1,log,45.297178,qty_rolling_mean_7,qty_rolling_mean_14,day_of_week,1000,The model mainly relies on historical SKU dema...


In [17]:
print("Model results files:")
for path in sorted(MODEL_RESULTS_DIR.glob("*shap*")):
    print(path)

print("\nFigure files:")
for path in sorted(FIGURES_DIR.glob("shap*.png")):
    print(path)

Model results files:
..\05_outputs\model_results\business_shap_summary.csv
..\05_outputs\model_results\local_shap_explanation.csv
..\05_outputs\model_results\shap_feature_importance.csv

Figure files:
..\05_outputs\figures\shap_bar_plot.png
..\05_outputs\figures\shap_summary_plot.png
